In [1]:
# pip install langchain_huggingface

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS,Chroma # FAISS ,Chroma --> types of VecorDB
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
import spacy
import re
import contractions
from textblob import TextBlob
import os
import string

C:\Users\ASUS\AppData\Local\Temp\ipykernel_29144\2273193693.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS,Chroma # FAISS ,Chroma --> types of VecorDB


In [3]:
data = TextLoader('rag.txt').load()

In [4]:
data = data[0].page_content.lower()

In [5]:
print(data)

what is rag?
 - it stands for retrieval augmented generation,it is an ai technique which retieives the relevant data from the external sources 
 and llm generates structured output

 - rag we use for custom chatbots
example :
customer support chatbots

why not llm?
- it is a foundation model which is already trained with vast amount of data to perform text generation tasks.
- it is not having all the information

rag architecture : 

1) data loader ?
- loading the informations from the external sources like pdf,word,text,csv.

2) text normalization ?
- cleaning or removing unwanted data from the document.
- this helps to reduce the tokens and make rag model to understand the context to generate proper result.

3) chuncking ?
it converts or splits the document into small parts (chuncks) 

why chuncking?
- rag model can't understand the context if load the entire document(content)
- it helps rag model to retrieve the actual context based on user query.

4) embeddings ?
- it is a way of c

In [6]:
var = string.punctuation
data = re.sub(f'[{var}]',' ',data)

In [7]:
data = re.sub(r'\s{2,}','',data)

## Lemmatization

In [8]:
nlp = spacy.load('en_core_web_sm')
tokens = nlp(data)
tokens

what is ragit stands for retrieval augmented generation it is an ai technique which retieives the relevant data from the external sourcesand llm generates structured outputrag we use for custom chatbots
examplecustomer support chatbotswhy not llmit is a foundation model which is already trained with vast amount of data to perform text generation tasksit is not having all the informationrag architecture1data loaderloading the informations from the external sources like pdf word text csv2text normalizationcleaning or removing unwanted data from the documentthis helps to reduce the tokens and make rag model to understand the context to generate proper result3chunckingit converts or splits the document into small partschunckswhy chunckingrag model can t understand the context if load the entire document contentit helps rag model to retrieve the actual context based on user query4embeddingsit is a way of converting chuncks into vectors numerical representation of chuncksrag model can t unde

In [9]:
updated_tokens = [str(token) for token in tokens if not token.is_stop] # is_stop--> returns boolean if the word is stopword
data = " ".join(updated_tokens)
print(data)
print(updated_tokens)

ragit stands retrieval augmented generation ai technique retieives relevant data external sourcesand llm generates structured outputrag use custom chatbots 
 examplecustomer support chatbotswhy llmit foundation model trained vast data perform text generation tasksit having informationrag architecture1data loaderloading informations external sources like pdf word text csv2text normalizationcleaning removing unwanted data documentthis helps reduce tokens rag model understand context generate proper result3chunckingit converts splits document small partschunckswhy chunckingrag model t understand context load entire document contentit helps rag model retrieve actual context based user query4embeddingsit way converting chuncks vectors numerical representation chuncksrag model t understand context text formatdirectlyso converting chuncks vectorshere usesentence transformers modelfor vectorization andit usesencoder onlytransformer architecturetransformers deep learning model architecture5vect

## **Chuncking**

In [10]:
'''.
Major Types of Chuncking,
- Sentence level Chuncking
- Word level Chuncking
- Fixed size Chuncking
chunck_size = 100
- Chunck Overlap
- Recursive Chuncking
- Semantic Chuncking
'''

'.\nMajor Types of Chuncking,\n- Sentence level Chuncking\n- Word level Chuncking\n- Fixed size Chuncking\nchunck_size = 100\n- Chunck Overlap\n- Recursive Chuncking\n- Semantic Chuncking\n'

In [11]:
# # Fixed Size Chuncking
# s1 = data
# s2 = []
# i = 0
# while i <= len(data):
#     for word in s1:
#         i+=10
#         word = data[:i]
#         s2.append(word)
#         break
# print(s2)

In [12]:
# Fixed size Chuncking
chunk_size = 150
s2 = []
for index in range(0,len(data),chunk_size):
    chunk = data[index:index+chunk_size]
    s2.append(chunk)
print(s2)

['ragit stands retrieval augmented generation ai technique retieives relevant data external sourcesand llm generates structured outputrag use custom cha', 'tbots \n examplecustomer support chatbotswhy llmit foundation model trained vast data perform text generation tasksit having informationrag architectur', 'e1data loaderloading informations external sources like pdf word text csv2text normalizationcleaning removing unwanted data documentthis helps reduce ', 'tokens rag model understand context generate proper result3chunckingit converts splits document small partschunckswhy chunckingrag model t understand ', 'context load entire document contentit helps rag model retrieve actual context based user query4embeddingsit way converting chuncks vectors numerical ', 'representation chuncksrag model t understand context text formatdirectlyso converting chuncks vectorshere usesentence transformers modelfor vectorizat', 'ion andit usesencoder onlytransformer architecturetransformers deep learn

In [13]:
# Chunk Overlap --> It must only 20 to 25% of chunk size 
'''.
chunk_size = 15
chunk_overlap = 7

chunk 1 => what is ragit s
chunk 2 => ragit stands fo
'''
chunk_size = 15
chunk_overlap = 5
s2 = []
for index in range(0,len(data),chunk_size-chunk_overlap):
    chunk = data[index:index+chunk_size]
    s2.append(chunk)
print(s2)

['ragit stands re', 'ds retrieval au', 'al augmented ge', 'ed generation a', 'ion ai techniqu', 'hnique retieive', 'ieives relevant', 'evant data exte', ' external sourc', 'sourcesand llm ', ' llm generates ', 'ates structured', 'tured outputrag', 'utrag use custo', 'custom chatbots', 'tbots \n example', 'amplecustomer s', 'mer support cha', 't chatbotswhy l', 'why llmit found', 'foundation mode', ' model trained ', 'ined vast data ', 'data perform te', 'rm text generat', 'neration tasksi', 'asksit having i', 'ing information', 'ationrag archit', 'rchitecture1dat', 'e1data loaderlo', 'derloading info', ' informations e', 'ons external so', 'al sources like', ' like pdf word ', 'word text csv2t', 'csv2text normal', 'ormalizationcle', 'oncleaning remo', ' removing unwan', 'unwanted data d', 'ata documentthi', 'ntthis helps re', 'ps reduce token', 'tokens rag mode', ' model understa', 'erstand context', 'ntext generate ', 'rate proper res', 'r result3chunck', 'hunckingit conv', ' converts

In [14]:
# Recursive Chuncking --> works based on seperator parameter 
# word, char --> overlap

In [15]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 100,
    separators=['\n\n','.','!','\n',' ','']
)
chunks = splitter.create_documents([data])
for chunk in chunks:
    print(chunk.page_content)

ragit stands retrieval augmented generation ai technique retieives relevant data external sourcesand llm generates structured outputrag use custom chatbots
examplecustomer support chatbotswhy llmit foundation model trained vast data perform text generation tasksit having informationrag architecture1data loaderloading informations external sources like
tasksit having informationrag architecture1data loaderloading informations external sources like pdf word text csv2text normalizationcleaning removing unwanted data documentthis helps reduce tokens
word text csv2text normalizationcleaning removing unwanted data documentthis helps reduce tokens rag model understand context generate proper result3chunckingit converts splits document small
rag model understand context generate proper result3chunckingit converts splits document small partschunckswhy chunckingrag model t understand context load entire document contentit helps rag model
chunckingrag model t understand context load entire docume

In [16]:
chunks = splitter.create_documents([data])
chunks[0].metadata={'file_name' : 'rag.txt'}
chunks
# i = 0
# while i < len(chunks):
#     chunks[i].metadata={'file_name' : 'rag.txt'}
#     i+=1
# print(chunks)
# for i in range(len(chunks)):
#     chunks[i].metadata={'file_name' : 'rag.txt'}
# chunk   
    

[Document(metadata={'file_name': 'rag.txt'}, page_content='ragit stands retrieval augmented generation ai technique retieives relevant data external sourcesand llm generates structured outputrag use custom chatbots'),
 Document(metadata={}, page_content='examplecustomer support chatbotswhy llmit foundation model trained vast data perform text generation tasksit having informationrag architecture1data loaderloading informations external sources like'),
 Document(metadata={}, page_content='tasksit having informationrag architecture1data loaderloading informations external sources like pdf word text csv2text normalizationcleaning removing unwanted data documentthis helps reduce tokens'),
 Document(metadata={}, page_content='word text csv2text normalizationcleaning removing unwanted data documentthis helps reduce tokens rag model understand context generate proper result3chunckingit converts splits document small'),
 Document(metadata={}, page_content='rag model understand context generate

In [17]:
# pip install sentence-transformers

### Transformer Architecture -- Deep learning
* sentence transformers --> Encoder only transformer architecture

In [18]:
embedding_model = HuggingFaceEmbeddings(
    model_name = 'sentence-transformers/all-miniLM-L6-V2' # sentence transformer - Encoder only - transformer architecture
)
embedding_model

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

HuggingFaceEmbeddings(model_name='sentence-transformers/all-miniLM-L6-V2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

## Chunks into Vectors

In [19]:
vectordb = FAISS.from_documents(    #FAISS ,Chroma --> VectorDB
    documents=chunks,
    embedding=embedding_model
)
vectordb # it has the information of embedding model 
# Vectors --> Numerical Representation of chunks

In [20]:
# pip install faiss-cup

In [21]:
user_query = "What is RAG ?"
r_chunks = vectordb.similarity_search(user_query) # Here it uses Euclidean Distance formula 
r_chunks
# similarity_search --> it has k => based on this it gives the chunk

[Document(id='1e4e8452-682a-4683-92c7-4b1dd491d768', metadata={}, page_content='rag model understand context generate proper result3chunckingit converts splits document small partschunckswhy chunckingrag model t understand context load entire document contentit helps rag model'),
 Document(id='d937c679-4cad-4d44-bce2-566494ca5669', metadata={}, page_content='chunckingrag model t understand context load entire document contentit helps rag model retrieve actual context based user query4embeddingsit way converting chuncks vectors numerical representation'),
 Document(id='5fdf1400-d192-47db-8cba-05b09c7e8ed5', metadata={}, page_content='word text csv2text normalizationcleaning removing unwanted data documentthis helps reduce tokens rag model understand context generate proper result3chunckingit converts splits document small'),
 Document(id='75ceca21-2595-467d-b378-8765ce19e024', metadata={}, page_content='required structure text outputwhat langchainit framework llm python library creating